# 混合数据集

![](./xiaodongguaAIGC_dataset.png)

In [42]:
from datasets import load_dataset, concatenate_datasets, DatasetDict

data_name1 = 'xiaodongguaAIGC/alpaca_gpt4_data_zh' # ignore
data_name2 = 'vicgalle/alpaca-gpt4'
data_name3 = 'LooksJuicy/ruozhiba'
data_name4 = 'silk-road/alpaca-data-gpt4-chinese' # 基于google翻译 仅演示如何按照比例来切割数据集

dataset1 = load_dataset(data_name1)
dataset2 = load_dataset(data_name2)
dataset3 = load_dataset(data_name3)
dataset4 = load_dataset(data_name4)

# dataset1: 中文alpaca数据集

In [43]:
dataset1 = load_dataset(data_name1)
print(dataset1)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 48818
    })
})

# dataset2: 英文 alpaca 数据集

In [44]:
# process dataset2
print(dataset2)
dataset2 = dataset2.remove_columns([
    'text',
])
print(dataset2)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 52002
    })
})

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 52002
    })
})

# dataset3:  弱智吧QA

In [45]:
# process dataset3
print(dataset3)
def process_ruozhiba(examples):
    examples['input'] = ''
    return examples
dataset3 = dataset3.map(process_ruozhiba, num_proc=8)
print(dataset3)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'output'],
        num_rows: 1496
    })
})

DatasetDict({
    train: Dataset({
        features: ['instruction', 'output', 'input'],
        num_rows: 1496
    })
})

# dataset4:  另一个alpaca中文数据

数据质量差不加入

In [46]:
print(dataset4)
def process_fn(examples):
    examples['instruction']=examples['instruction_zh']
    examples['input']=examples['input_zh']
    examples['output']=examples['output_zh']
    return examples
dataset4 = dataset4.map(process_fn, num_proc=8, remove_columns = ["instruction_zh", "input_zh", 'output_zh'])
dataset4['train'] = dataset4['train'].shard(num_shards=10, index=0) # 只取10%数据
print(dataset4)

DatasetDict({
    train: Dataset({
        features: ['instruction_zh', 'input_zh', 'output_zh', 'instruction', 'input', 'output'],
        num_rows: 52049
    })
})

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 5205
    })
})

In [47]:
# 规则过滤: 基于长度过滤
print(dataset4)
dataset4['train'] = dataset4['train'].filter(lambda example: len(example["instruction"]+
                                                                 example["input"]+
                                                                 example["output"])<512)
print(dataset4)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 5205
    })
})

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 4670
    })
})

# dataset5. 英文选择题 allenai/ai2_arc

In [48]:
data_name5 = 'allenai/ai2_arc'
dataset5 = load_dataset(data_name5, 'ARC-Challenge')
dataset6 = load_dataset(data_name5, 'ARC-Easy')
print(dataset5)
print(dataset6)

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 1119
    })
    test: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 1172
    })
    validation: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 299
    })
})

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 2251
    })
    test: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 2376
    })
    validation: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 570
    })
})

In [49]:
print(dataset5['train'][0])

{
    'id': 'Mercury_SC_415702',
    'question': 'George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most 
heat?',
    'choices': {
        'text': ['dry palms', 'wet palms', 'palms covered with oil', 'palms covered with lotion'],
        'label': ['A', 'B', 'C', 'D']
    },
    'answerKey': 'A'
}

In [50]:
mcq_prompt = 'There is a single choice question. Answer the question by replying A, B, C or D.\n'
def process_arc(examples):
    
    prompt_q = 'Question: ' + examples['question'] 
    for label, text  in zip(examples['choices']['label'], examples['choices']['text']):
               prompt_q = prompt_q + label + '. ' + text + '\n'  
    prompt_a = examples['answerKey'] 
    examples['instruction'] = mcq_prompt
    examples['input'] = prompt_q
    examples['output'] = prompt_a
    return examples


dataset5_alpaca = dataset5.map(process_arc)
print(dataset5_alpaca)
dataset6_alpaca = dataset6.map(process_arc)
print(dataset6_alpaca)
    

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'choices', 'answerKey', 'instruction', 'input', 'output'],
        num_rows: 1119
    })
    test: Dataset({
        features: ['id', 'question', 'choices', 'answerKey', 'instruction', 'input', 'output'],
        num_rows: 1172
    })
    validation: Dataset({
        features: ['id', 'question', 'choices', 'answerKey', 'instruction', 'input', 'output'],
        num_rows: 299
    })
})

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'choices', 'answerKey', 'instruction', 'input', 'output'],
        num_rows: 2251
    })
    test: Dataset({
        features: ['id', 'question', 'choices', 'answerKey', 'instruction', 'input', 'output'],
        num_rows: 2376
    })
    validation: Dataset({
        features: ['id', 'question', 'choices', 'answerKey', 'instruction', 'input', 'output'],
        num_rows: 570
    })
})

In [51]:
print(dataset5_alpaca['train'][0])

{
    'id': 'Mercury_SC_415702',
    'question': 'George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most 
heat?',
    'choices': {
        'text': ['dry palms', 'wet palms', 'palms covered with oil', 'palms covered with lotion'],
        'label': ['A', 'B', 'C', 'D']
    },
    'answerKey': 'A',
    'instruction': 'There is a single choice question. Answer the question by replying A, B, C or D.\n',
    'input': 'Question: George wants to warm his hands quickly by rubbing them. Which skin surface will produce the
most heat?A. dry palms\nB. wet palms\nC. palms covered with oil\nD. palms covered with lotion\n',
    'output': 'A'
}

In [52]:
dataset5_alpaca['train'] = dataset5_alpaca['train'].filter(lambda example: len(example["choices"]['label'])==4 )
dataset5_alpaca['test'] = dataset5_alpaca['test'].filter(lambda example: len(example["choices"]['label'])==4 )
dataset5_alpaca['validation'] = dataset5_alpaca['validation'].filter(lambda example: len(example["choices"]['label'])==4 )
dataset6_alpaca['train'] = dataset6_alpaca['train'].filter(lambda example: len(example["choices"]['label'])==4 )
dataset6_alpaca['test'] = dataset6_alpaca['test'].filter(lambda example: len(example["choices"]['label'])==4 )
dataset6_alpaca['validation'] = dataset6_alpaca['validation'].filter(lambda example: len(example["choices"]['label'])==4 )
print(dataset5_alpaca)

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'choices', 'answerKey', 'instruction', 'input', 'output'],
        num_rows: 1117
    })
    test: Dataset({
        features: ['id', 'question', 'choices', 'answerKey', 'instruction', 'input', 'output'],
        num_rows: 1165
    })
    validation: Dataset({
        features: ['id', 'question', 'choices', 'answerKey', 'instruction', 'input', 'output'],
        num_rows: 295
    })
})

In [53]:
dataset6_alpaca = dataset6_alpaca.remove_columns(["id", "question", 'choices', 'answerKey'])
dataset5_alpaca = dataset5_alpaca.remove_columns(["id", "question", 'choices', 'answerKey'])
print(dataset5_alpaca)
print(dataset6_alpaca)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 1117
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 1165
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 295
    })
})

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 2241
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 2365
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 567
    })
})

# 7. 中文选择题+COT

In [54]:
# ZhangRC/chinese-multi-choice-ceval-validation-glm4-explanation
data_name7 = 'ZhangRC/chinese-multi-choice-ceval-validation-glm4-explanation'
dataset7 = load_dataset(data_name7)
print(dataset7)
print(dataset7['train'][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'A', 'B', 'C', 'D', 'answer', 'explanation', 'domain'],
        num_rows: 1176
    })
})

{
    'id': 0,
    'question': '下列关于税法基本原则的表述中，不正确的是____。',
    'A': '税收法定原则包括税收要件法定原则和税务合法性原则',
    'B': '税收公平原则源于法律上的平等性原则',
    'C': '税收效率原则包含经济效率和行政效率两个方面',
    'D': '税务机关按法定程序依法征税，可以自由做出减征、停征或免征税款的决定',
    'answer': 'D',
    'explanation': '**分析和解读：**\n\nA. **税收法定原则：** 
这是税法的一个基本原则，指的是税收征收必须有法律依据，包括税收的要件（如税率、税基、税收对象等）必须由法律规定，以
及税务行为必须符合法律程序。这个选项正确地表述了税收法定原则的两个方面：税收要件法定原则和税务合法性原则。\n\nB. 
**税收公平原则：** 
这个原则强调的是税收负担应根据纳税人的经济能力分配，即能力相同的人应承担相同的税负，能力不同的人应承担不同的税负。
它确实源于法律上的平等性原则，即相似的情况应得到相似的处理。\n\nC. **税收效率原则：** 
这个原则有两个方面：经济效率和行政效率。经济效率关注税收对经济活动的影响，应尽量减少对经济活动的扭曲；行政效率关注
税收管理的成本，应尽量降低税收征管的成本和提高税收征管效率。这个选项正确地描述了税收效率原则的两个方面。\n\nD. 
**税务机关的权力：** 
税务机关在征税过程中必须遵守法定程序，不能随意减征、停征或免征税款。任何税收减免都必须有明确的法律依据，税务机关没
有自由做出这些决定的权力。\n\n**权威正确答案：**\n\nD. 
税务机关按法定程序依法征税，可以自由做出减征、停征或免征税款的决定\n\n这个选项的表述是不正确的。根据税收法定原则，
税务机关必须依据法律的规定进行征税，不能自由做出减征、停征或免征税款的决定，这样的决定需要有明确的法律授权或依据。
因此，正确答案是D。',
    'domain': 'accountant'
}

In [55]:
mcq_prompt = '以下是单项选择题，请直接给出正确答案的选项A,B,C,D。\n'
def process_cmmlu(examples):
    prompt_q = '题目:' + examples['question'] + '\n' \
                + 'A. ' + examples['A'] + '\n' \
                + 'B. ' + examples['B'] + '\n' \
                + 'C. ' + examples['C'] + '\n' \
                + 'D. ' + examples['D'] + '\n' 
    # prompt_a = examples['answer'] 
    if examples['id']%10 == 0:
        prompt_a = examples['explanation']
    else:
        prompt_a = examples['answer']
    examples['instruction'] = mcq_prompt
    examples['input'] = prompt_q
    examples['output'] = prompt_a
    return examples
    
dataset7_alpaca = dataset7.map(process_cmmlu, remove_columns=['id','question','A','B','C','D','answer','explanation','domain'])
print(dataset7_alpaca)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 1176
    })
})

In [56]:
print(dataset7_alpaca['train'][0]) # CoT
print(dataset7_alpaca['train'][1])

{
    'instruction': '以下是单项选择题，请直接给出正确答案的选项A,B,C,D。\n',
    'input': '题目:下列关于税法基本原则的表述中，不正确的是____。\nA. 
税收法定原则包括税收要件法定原则和税务合法性原则\nB. 税收公平原则源于法律上的平等性原则\nC. 
税收效率原则包含经济效率和行政效率两个方面\nD. 
税务机关按法定程序依法征税，可以自由做出减征、停征或免征税款的决定\n',
    'output': '**分析和解读：**\n\nA. **税收法定原则：** 
这是税法的一个基本原则，指的是税收征收必须有法律依据，包括税收的要件（如税率、税基、税收对象等）必须由法律规定，以
及税务行为必须符合法律程序。这个选项正确地表述了税收法定原则的两个方面：税收要件法定原则和税务合法性原则。\n\nB. 
**税收公平原则：** 
这个原则强调的是税收负担应根据纳税人的经济能力分配，即能力相同的人应承担相同的税负，能力不同的人应承担不同的税负。
它确实源于法律上的平等性原则，即相似的情况应得到相似的处理。\n\nC. **税收效率原则：** 
这个原则有两个方面：经济效率和行政效率。经济效率关注税收对经济活动的影响，应尽量减少对经济活动的扭曲；行政效率关注
税收管理的成本，应尽量降低税收征管的成本和提高税收征管效率。这个选项正确地描述了税收效率原则的两个方面。\n\nD. 
**税务机关的权力：** 
税务机关在征税过程中必须遵守法定程序，不能随意减征、停征或免征税款。任何税收减免都必须有明确的法律依据，税务机关没
有自由做出这些决定的权力。\n\n**权威正确答案：**\n\nD. 
税务机关按法定程序依法征税，可以自由做出减征、停征或免征税款的决定\n\n这个选项的表述是不正确的。根据税收法定原则，
税务机关必须依据法律的规定进行征税，不能自由做出减征、停征或免征税款的决定，这样的决定需要有明确的法律授权或依据。
因此，正确答案是D。'
}

{
    'instruction': '以下是单项选择题，请直接给出正确答案的选项A,B,C,D。\n',
    'input': 
'题目:甲公司是国内一家领先的新媒体、通信及移动增值服务公司，由于遭受世界金融危机，甲公司经济利润严重下滑，经营面临
困境，但为了稳定职工队伍，公司并未进行裁员，而是实行高层管理人员减薪措施。甲公司此举采用的收缩战略方式是____。\nA. 
转向战略\nB. 放弃战略\nC. 紧缩与集中战略\nD. 稳定战略\n',
    'output': 'C'
}

# 合并数据集

In [57]:
dataset = concatenate_datasets([dataset1['train'], dataset2['train'], dataset3['train'],
                               dataset5_alpaca['train'], dataset5_alpaca['validation'], dataset5_alpaca['test'],
                               dataset6_alpaca['train'], dataset6_alpaca['validation'], dataset6_alpaca['test'],
                               dataset7_alpaca['train'] ])
dataset = DatasetDict({'train': dataset})
dataset = dataset.shuffle(seed=42)
print(dataset)
# print(dataset['train'][:20])
for i in range(20):
    print('-'*100)
    print('instruction: ', dataset['train']['instruction'][i])
    print('input: ', dataset['train']['input'][i])
    print('output: ', dataset['train']['output'][i])

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 111242
    })
})

----------------------------------------------------------------------------------------------------

instruction:  Describe how a GPT model works.

input:

output:  GPT, or Generative Pre-trained Transformer, is a type of neural network-based language model that is 
designed to generate human-like text. It works by predicting the next word in a sequence based on the words that 
have come before it, using a large dataset of text to train its predictions.

The GPT model first preprocesses the input text into a format known as tokenization, where the text is broken down 
into individual units called tokens. These tokens are then fed into the model's architecture, which consists of 
multiple layers of interconnected nodes called neurons. At each layer, the model performs various mathematical 
operations to transform the input and maps it to a higher-level representation.

A key component of the GPT model is its attention mechanism. This allows the model to weigh the importance of each 
word in the sequence when making its predictions. For instance, the model can learn that the word "dog" is more 
likely to appear after "a furry animal that barks" than after "a delicious dessert." This is done by calculating 
the relationship between each word and every other word in the sequence, to determine which words are most relevant
for making the current prediction.

After passing through multiple layers of the model, the final output is a probability distribution over all the 
possible next words in the vocabulary. The model selects the most likely word as its prediction and adds it to the 
sequence. With each additional word, the model updates its internal state and makes its next prediction based on 
the updated context.

GPT models can be trained on a wide range of text data, from Wikipedia articles to books, and can be fine-tuned for
specific tasks, such as generating realistic news articles or completing sentences given a prompt. Their ability to
generate coherent and human-like text has many applications, including language translation, text summarization, 
and chatbots.

...

output:  作为一个语言模型，我无法直接创作音乐。但是，我可以提供一些建议和结构来帮助你创作一首两分钟的钢琴曲:

1. 结构与进展: 
一首两分钟的钢琴曲可能包含两到三个主要部分。比如，你可以以一个平缓的节奏开始，当音乐渐渐升级，然后加入一些快速和激
烈的音符来增加情感亮度与张力。

2. 和弦与旋律: 
选择一个基础调性来构建你的和弦进行和旋律。这可以是一个大调，也可以是一个小调，具体取决于你想要创作的音乐风格和情感
。

3. 节奏与时间: 
在两分钟的时间内，使用不同的节奏和时间感来增加曲子的多样性。可以使用快节奏来营造紧张和兴奋感，也可以采用较慢的节奏
来传达温柔和平静的情绪。

4. 表现力与动态: 
使用刚强、弱音、渐强渐弱、断音等各种表现力手法来具体表达音乐中的情感。例如，可以使用强音来增强关键部分的张力，或使
用弱音来营造温柔的氛围。

在以上的建议中，你可以根据你的灵感和创造力来创作一首独特的两分钟钢琴曲。

----------------------------------------------------------------------------------------------------

instruction:  Find the product of 9, 5 and 28.

input:

output:  The product of 9, 5, and 28 is 1260.

# 保存

In [58]:
dataset.save_to_disk('./output/alpaca_en_zh_ruozhiba')

Saving the dataset (0/1 shards):   0%|          | 0/111242 [00:00<?, ? examples/s]